# Course: 65007 - NLP and Speech Analysis
 **Program:** Intelligent Systems  
 **Course coordinator:** Dr. Sharon Yalov-Handzel

**Submission for:**  Assignment 2   
**by**  
  - Michael Berger, 318063864  
  - Barack Samuni, 318299625

# 1. Word2Vec

## a. Write Python program to implement Skip-gram Word2Vec algorithm.
Barak

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize

# Download necessary NLTK resources
nltk.download('punkt')


class SkipGramDataset(Dataset):
    def __init__(self, tokenized_corpus, word_to_index, window_size):
        """
        Custom dataset for Skip-gram Word2Vec.
        Generates target-context word pairs based on the pre-tokenized corpus.
        """
        self.pairs = []  # Store target-context pairs

        # Generate target-context pairs
        for sentence in tokenized_corpus:
            sentence_indices = [word_to_index[word] for word in sentence]
            for i, target_index in enumerate(sentence_indices):
                # Create a context window around the target word
                context_indices = sentence_indices[max(0, i - window_size):i] + \
                                  sentence_indices[i + 1:min(len(sentence_indices), i + window_size + 1)]
                for context_index in context_indices:
                    # Append the (target, context) pair to the list
                    self.pairs.append((target_index, context_index))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        return self.pairs[index]


class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        """
        Skip-gram Word2Vec model with trainable embeddings.
        """
        super(SkipGramModel, self).__init__()
        self.input_embeddings = nn.Embedding(vocab_size, embedding_size)    # target words to dense embedding vectors
        self.output_embeddings = nn.Embedding(vocab_size, embedding_size)   # context words to dense embedding vectors

    def forward(self, target_words, context_words):
        """
        Forward pass to compute the logits (scores) for target-context word pairs.
        """
        # Embedding lookup for target and context words
        target_embeds = self.input_embeddings(target_words)     # Shape: (batch_size, embedding_size)
        context_embeds = self.output_embeddings(context_words)  # Shape: (batch_size, embedding_size)

        # Compute dot product (logits) between target and context embeddings
        logits = torch.sum(target_embeds * context_embeds, dim=1)  # Shape: (batch_size)

        return logits

def train_skipgram_model(corpus, embedding_size=10, window_size=2, learning_rate=0.01, epochs=10, batch_size=64):
    """
    Trains a Skip-gram Word2Vec model with PyTorch. Handles tokenization and vocab generation internally.
    """
    # Tokenization and Vocabulary Creation
    tokenized_corpus = [word_tokenize(sentence.lower()) for sentence in corpus]
    words = [word for sentence in tokenized_corpus for word in sentence]
    vocab = list(set(words))
    word_to_index = {word: i for i, word in enumerate(vocab)}
    index_to_word = {i: word for word, i in word_to_index.items()}
    vocab_size = len(word_to_index)

    # Create Skip-gram dataset and dataloader
    dataset = SkipGramDataset(tokenized_corpus, word_to_index, window_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Initialize the SkipGram model
    model = SkipGramModel(vocab_size, embedding_size)
    criterion = nn.CrossEntropyLoss()  # Negative log likelihood loss with softmax
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Get the training device (CPU or GPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training device: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")
    model.to(device)

    # Training loop
    for epoch in range(epochs):
        total_loss = 0
        for target, context in dataloader:
            # Send tensors to the training device
            target = target.to(device)
            context = context.to(device)

            # Forward pass
            logits = model.input_embeddings(target) @ model.output_embeddings.weight.T  # Shape: (batch_size, vocab_size)

            # Compute loss
            loss = criterion(logits, context)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss:.4f}")

    # Extract word embeddings to a dictionary
    embeddings = model.input_embeddings.weight.cpu().detach().numpy()
    embedding_dict = {index_to_word[i]: embeddings[i] for i in range(vocab_size)}

    return embedding_dict

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\barak\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
## b. Write Python program that implements CBOW Word2Vec algorithm
Michael

## c. Apply both programs to the following text:

```
i. The bank is located near the river.
ii. The bank approved my loan application.
iii. He rose from his chair to close the window.
iv. The rose bloomed beautifully in the garden.
v. The lead actor delivered a stunning performance.
vi. Exposure to lead is harmful to health.
vii. She is reading a book in the library.
viii. The book mentioned a fascinating historical event.
ix. I need to file a report for my manager.
x. He lost the file containing important documents.
```

Skip-gram

In [10]:
corpus = [
    "The bank is located near the river.",
    "The bank approved my loan application.",
    "He rose from his chair to close the window.",
    "The rose bloomed beautifully in the garden.",
    "The lead actor delivered a stunning performance.",
    "Exposure to lead is harmful to health.",
    "She is reading a book in the library.",
    "The book mentioned a fascinating historical event.",
    "I need to file a report for my manager.",
    "He lost the file containing important documents."
]
embedding_dict_skipgram = train_skipgram_model(corpus)
embedding_dict_skipgram

Training device: GPU (CUDA)
Epoch 1/10, Loss: 36.8129
Epoch 2/10, Loss: 34.0520
Epoch 3/10, Loss: 32.1559
Epoch 4/10, Loss: 30.3851
Epoch 5/10, Loss: 29.1702
Epoch 6/10, Loss: 27.4911
Epoch 7/10, Loss: 26.6007
Epoch 8/10, Loss: 25.3143
Epoch 9/10, Loss: 24.2665
Epoch 10/10, Loss: 23.6379


{'he': array([-1.2009562 ,  0.09446239, -0.62522995,  0.05828322, -0.26441813,
         0.26531664,  0.390266  , -0.3942018 ,  0.15618093,  0.26379827],
       dtype=float32),
 'mentioned': array([-0.18618171,  0.39249304,  1.3484132 ,  0.06792637, -0.9293208 ,
         1.5880983 ,  1.0234445 ,  1.1302633 ,  0.8156896 , -1.5344201 ],
       dtype=float32),
 'reading': array([ 1.1067728 , -0.2366533 , -1.2428963 ,  0.1637473 , -0.6696501 ,
        -0.30681115, -0.79040825, -0.8601573 ,  0.44870508,  0.00859739],
       dtype=float32),
 'a': array([-0.43903244, -1.7169126 , -1.1258068 ,  0.45893404, -1.0679713 ,
        -0.46620524,  0.5493672 ,  0.3674028 ,  0.9452898 ,  0.64792705],
       dtype=float32),
 'is': array([ 0.5592894 , -0.05521967, -0.5305127 ,  0.08293814,  0.83313644,
        -2.0258982 , -1.1458248 ,  0.29426628, -0.542919  , -0.0677876 ],
       dtype=float32),
 'she': array([ 0.01218892,  1.2505721 , -0.78648984,  0.91646934,  0.5630935 ,
        -0.7146004 , -2.11086

CBOW

## d. What is the difference between the embeddings? 
Explain the results.

## e. Can you find a text that 
Its embedding will be similar in these two algorithms?

## f. Repeat step c with different window sizes. 
Is there a significant change?

### Skip-gram

### CBOW

## g. Compare these two models 
In terms of capturing the syntactic and the semantic relationship between words.

## h. Demonstrate the difference between CBOW and Skip-grams
In terms of cosine similarity between the following words:  
 - bank, rose, lead, book and file.

### Skip-gram

### CBOW

## i. How can the subword embeddings be applied?
Michael

# 2. Create example sentences demonstrating: 
how contextual embeddings handle words with multiple meanings (polysemy) differently than static embeddings like Word2Vec.
Barak

# 3. Propose metrics
For evaluating word embeddings that can differentiate between syntactic and semantic relationships.
Michael

# 4. Use the Gensim library 
To train a Word2Vec model on a custom corpus.  
Barak

## a. Evaluate the quality of embeddings 
By calculating the cosine similarity for the following word pairs:
```
i. "king" and "queen"  
ii. "man" and "woman"
iii. "apple" and "orange"
```

## b. Write a brief explanation of the results.

# 5. Train the GloVe model 
Using the glove-python package on a subset of a publicly available dataset (e.g., Wikipedia, or a smaller custom corpus).

Michael

## a. Use t-SNE or PCA 
To visualize the embeddings in 2D.

## b. Analyze the clustering patterns observed in the visualization.